# GeoRoad Inspector — YOLOv8 Training on RDD2022

Fine-tunes **YOLOv8s** on RDD2022 for 4 road damage classes.

**Before running:** Runtime → Change runtime type → T4 GPU

Expected time: ~15–25 min on T4 GPU.

## Step 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Install dependencies

In [ ]:
!pip install ultralytics requests tqdm Pillow -q
import ultralytics
print('Ultralytics:', ultralytics.__version__)

## Step 3 — Download RDD2022

In [ ]:
import os, zipfile, requests
from tqdm import tqdm

os.makedirs('/content/rdd2022', exist_ok=True)

COUNTRY_URLS = {
    'Japan':         'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Japan.zip',
    'India':         'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip',
    'Czech':         'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Czech.zip',
    'United_States': 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_United_States.zip',
}

def download_file(url, dest):
    if os.path.exists(dest):
        print(f'Already downloaded: {dest}')
        return
    print(f'Downloading {os.path.basename(dest)}...')
    r = requests.get(url, stream=True)
    total = int(r.headers.get('content-length', 0))
    with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))

for country, url in COUNTRY_URLS.items():
    zip_path = f'/content/rdd2022/{country}.zip'
    download_file(url, zip_path)
    if not os.path.exists(f'/content/rdd2022/{country}'):
        print(f'Extracting {country}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/rdd2022/')

print('Done.')

## Step 4 — Diagnose actual file structure

Run this to see real filenames before conversion.

In [ ]:
import glob

for country in ['Japan', 'India', 'Czech', 'United_States']:
    base = f'/content/rdd2022/{country}'
    if not os.path.exists(base):
        print(f'{country}: NOT FOUND')
        continue

    # find all jpgs and xmls
    jpgs = glob.glob(f'{base}/**/*.jpg', recursive=True)
    xmls = glob.glob(f'{base}/**/*.xml', recursive=True)
    print(f'{country}: {len(jpgs)} images, {len(xmls)} xmls')
    if jpgs:
        print(f'  Sample image path: {jpgs[0]}')
        print(f'  Image filename:    {os.path.basename(jpgs[0])}')
    if xmls:
        print(f'  Sample xml path:   {xmls[0]}')
        print(f'  XML filename:      {os.path.basename(xmls[0])}')
    print()

## Step 5 — Convert Pascal VOC → YOLO

Uses pure glob-based matching — no hardcoded paths.

In [ ]:
import glob, shutil, random
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image as PILImage

CLASS_MAP = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}

for split in ['train', 'val']:
    os.makedirs(f'/content/dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/dataset/labels/{split}', exist_ok=True)

def convert_voc_to_yolo(xml_path, img_w, img_h):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception:
        return []
    lines = []
    for obj in root.findall('object'):
        name_el = obj.find('name')
        if name_el is None:
            continue
        name = name_el.text
        if name not in CLASS_MAP:
            continue
        cls = CLASS_MAP[name]
        bb = obj.find('bndbox')
        if bb is None:
            continue
        try:
            xmin = float(bb.find('xmin').text)
            ymin = float(bb.find('ymin').text)
            xmax = float(bb.find('xmax').text)
            ymax = float(bb.find('ymax').text)
        except Exception:
            continue
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines

all_samples = []

for country in ['Japan', 'India', 'Czech', 'United_States']:
    base = f'/content/rdd2022/{country}'
    if not os.path.exists(base):
        print(f'Skipping {country} — not found')
        continue

    # Build a stem → xml_path lookup from ALL xmls under this country
    all_xmls = glob.glob(f'{base}/**/*.xml', recursive=True)
    xml_by_stem = {Path(x).stem: x for x in all_xmls}

    # Find all training images (skip /test/ directories)
    all_jpgs = glob.glob(f'{base}/**/*.jpg', recursive=True)
    train_jpgs = [p for p in all_jpgs if '/test/' not in p and '/Test/' not in p]

    count_before = len(all_samples)
    for img_path in train_jpgs:
        stem = Path(img_path).stem
        xml_path = xml_by_stem.get(stem)
        if not xml_path:
            continue
        try:
            with PILImage.open(img_path) as im:
                w, h = im.size
        except Exception:
            continue
        yolo_lines = convert_voc_to_yolo(xml_path, w, h)
        if not yolo_lines:
            continue
        all_samples.append((img_path, yolo_lines, stem))

    added = len(all_samples) - count_before
    print(f'{country}: +{added} samples (total xmls={len(all_xmls)}, total train imgs={len(train_jpgs)})')

print(f'\nTotal valid samples: {len(all_samples)}')
assert len(all_samples) > 0, 'No samples found — check Step 4 output'

random.seed(42)
random.shuffle(all_samples)
split_idx = int(len(all_samples) * 0.8)
train_samples = all_samples[:split_idx]
val_samples   = all_samples[split_idx:]

for img_path, yolo_lines, stem in train_samples:
    shutil.copy(img_path, f'/content/dataset/images/train/{stem}.jpg')
    with open(f'/content/dataset/labels/train/{stem}.txt', 'w') as f:
        f.write('\n'.join(yolo_lines))

for img_path, yolo_lines, stem in val_samples:
    shutil.copy(img_path, f'/content/dataset/images/val/{stem}.jpg')
    with open(f'/content/dataset/labels/val/{stem}.txt', 'w') as f:
        f.write('\n'.join(yolo_lines))

print(f'Train: {len(train_samples)} | Val: {len(val_samples)}')
print(f'Images written: {len(os.listdir("/content/dataset/images/train"))} train, {len(os.listdir("/content/dataset/images/val"))} val')

## Step 6 — Create dataset YAML

In [ ]:
yaml_content = 'path: /content/dataset\ntrain: images/train\nval: images/val\n\nnc: 4\nnames:\n  0: longitudinal_crack\n  1: transverse_crack\n  2: alligator_crack\n  3: pothole\n'

with open('/content/rdd2022.yaml', 'w') as f:
    f.write(yaml_content)

print('YAML ready:')
print(yaml_content)

## Step 7 — Train YOLOv8s (100 epochs)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/rdd2022.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    name='road_damage',
    project='/content/runs',
    exist_ok=True,
    pretrained=True,
)

print('Training complete!')
print('Best: /content/runs/road_damage/weights/best.pt')

## Step 8 — Evaluate

In [ ]:
from ultralytics import YOLO
best = YOLO('/content/runs/road_damage/weights/best.pt')
m = best.val(data='/content/rdd2022.yaml', imgsz=640)
print('=== Validation Metrics ===')
print(f'Precision:  {m.box.mp*100:.1f}%')
print(f'Recall:     {m.box.mr*100:.1f}%')
print(f'mAP@50:     {m.box.map50*100:.1f}%')
print(f'mAP@50-95:  {m.box.map*100:.1f}%')
for cls, ap in zip(['longitudinal_crack','transverse_crack','alligator_crack','pothole'], m.box.ap50):
    print(f'  {cls}: {ap*100:.1f}%')

## Step 9 — Download model

Save as `backend/models/road_damage_best.pt` in your repo.

In [ ]:
import shutil
shutil.copy('/content/runs/road_damage/weights/best.pt', '/content/road_damage_best.pt')
from google.colab import files
files.download('/content/road_damage_best.pt')
print('Downloading road_damage_best.pt')

## Step 10 — (Optional) Push to GitHub directly

In [ ]:
GITHUB_TOKEN = ''  # paste your GitHub PAT with repo write scope
REPO = 'manumanoj0010/georoad-inspector'
BRANCH = 'development'

if GITHUB_TOKEN:
    import shutil, subprocess
    subprocess.run(['git','clone',f'https://{GITHUB_TOKEN}@github.com/{REPO}.git','/content/repo'], check=True)
    shutil.copy('/content/road_damage_best.pt', '/content/repo/backend/models/road_damage_best.pt')
    subprocess.run(['git','-C','/content/repo','config','user.email','training@georoad.ai'])
    subprocess.run(['git','-C','/content/repo','config','user.name','GeoRoad Training'])
    subprocess.run(['git','-C','/content/repo','add','backend/models/road_damage_best.pt'])
    subprocess.run(['git','-C','/content/repo','commit','-m','Update model: YOLOv8s 100 epochs RDD2022'])
    subprocess.run(['git','-C','/content/repo','push','origin',BRANCH])
    print('Pushed to GitHub!')
else:
    print('Skipped — use Step 9 to download manually')